In [5]:
import pandas as pd
from pathlib import Path

RAW = Path.cwd().parent / 'data' / '02_features' / 'raw'

for f in sorted(RAW.glob('*.csv')):
    df = pd.read_csv(f)
    print(f"\n=== {f.name} ===")
    print(f"Shape: {df.shape} | Columns: {df.columns.tolist()}")
    print(df.head(2).to_string())
    print(f"Date col dtype: {df.iloc[:,0].dtype}")


=== baa10y.csv ===
Shape: (10514, 2) | Columns: ['observation_date', 'BAA10Y']
  observation_date  BAA10Y
0       1986-01-02    2.34
1       1986-01-03    2.30
Date col dtype: object

=== bdi_clean.csv ===
Shape: (9479, 2) | Columns: ['date', 'bdi']
         date        bdi
0  1986-12-31  2568.3000
1  1987-01-02  2540.1001
Date col dtype: object

=== brent_fred.csv ===
Shape: (9874, 2) | Columns: ['observation_date', 'DCOILBRENTEU']
  observation_date  DCOILBRENTEU
0       1987-05-20         18.63
1       1987-05-21         18.45
Date col dtype: object

=== cny_usd.csv ===
Shape: (11816, 2) | Columns: ['observation_date', 'DEXCHUS']
  observation_date  DEXCHUS
0       1981-01-02   1.5341
1       1981-01-05   1.5418
Date col dtype: object

=== dxy.csv ===
Shape: (5295, 2) | Columns: ['observation_date', 'DTWEXBGS']
  observation_date  DTWEXBGS
0       2006-01-02  101.4155
1       2006-01-03  100.7558
Date col dtype: object

=== em_fx_idx.csv ===
Shape: (5295, 2) | Columns: ['observatio

In [6]:
from __future__ import annotations

from pathlib import Path
import re
from typing import Iterable

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from linearmodels.iv import IV2SLS
from statsmodels.regression.quantile_regression import QuantReg
from statsmodels.tools.tools import add_constant

HMAX = 48

# ── Paths ──────────────────────────────────────────────────────────────────────
cwd = Path.cwd().resolve()
ROOT = cwd.parent if cwd.name == 'notebooks' else cwd

ORIGINAL = ROOT / 'original'
FIGURES = ROOT / 'figures'
RESULTS = ROOT / 'results'
CACHE = ROOT / 'data' / 'cache'
RAW = ROOT / 'data' / '02_features' / 'raw'
NLP = ROOT / 'data' / '03_nlp'
FINAL = ROOT / 'data' / 'final'

dta_data = ROOT / 'data' / 'Saadaoui_2026_JCE.dta'
dta_original = ORIGINAL / 'Saadaoui_2026_JCE.dta'
DTA = dta_data if dta_data.exists() else dta_original
LOG = ORIGINAL / 'Saadaoui_2026_JCE.log'

for d in [FIGURES, RESULTS, CACHE, FINAL]:
    d.mkdir(parents=True, exist_ok=True)

print(f'ROOT → {ROOT}')
print(f'DTA → {DTA} (exists: {DTA.exists()})')
print(f'RAW → {RAW} (exists: {RAW.exists()})')
print(f'NLP → {NLP} (exists: {NLP.exists()})')

ROOT → C:\Users\HP\Desktop\replication+contribution
DTA → C:\Users\HP\Desktop\replication+contribution\data\Saadaoui_2026_JCE.dta (exists: True)
RAW → C:\Users\HP\Desktop\replication+contribution\data\02_features\raw (exists: True)
NLP → C:\Users\HP\Desktop\replication+contribution\data\03_nlp (exists: True)


In [7]:
def D(s: pd.Series) -> pd.Series:
    return s.diff()

def F(s: pd.Series, h: int) -> pd.Series:
    return s.shift(-h)

def stata_month_to_datetime(period: pd.Series) -> pd.Series:
    if pd.api.types.is_datetime64_any_dtype(period):
        return pd.to_datetime(period)
    base = pd.Period('1960-01', freq='M')
    numeric = pd.to_numeric(period, errors='coerce')
    return numeric.map(
        lambda m: (base + int(m)).to_timestamp(how='end') if pd.notna(m) else pd.NaT
    )

def add_lagged_controls(df, y_col='lwti', shock_col='lpri', y_lags=3, shock_lags=2):
    out = df.copy()
    lag_cols = []
    for l in range(1, y_lags + 1):
        c = f'L{l}_{y_col}'
        out[c] = out[y_col].shift(l)
        lag_cols.append(c)
    for l in range(1, shock_lags + 1):
        c = f'L{l}_{shock_col}'
        out[c] = out[shock_col].shift(l)
        lag_cols.append(c)
    return out, lag_cols

def first_stage_f(df, x='lpri', z='d2pri', controls=None):
    if controls is None:
        controls = ['llwip', 'dllgop', 'l2lwip', 'dl2lgop']
    work, lag_cols = add_lagged_controls(df, y_col='lwti', shock_col=x, y_lags=3, shock_lags=2)
    exog_cols = lag_cols + controls
    fdf = pd.DataFrame({x: work[x], z: work[z], **{c: work[c] for c in exog_cols}}).dropna()
    X = add_constant(fdf[[z] + exog_cols], has_constant='add')
    fit = sm.OLS(fdf[x], X).fit(cov_type='HC1')
    return float(fit.f_test(f'{z} = 0').fvalue)

def to_monthly_index(df, date_col):
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col])
    df = df.set_index(date_col).sort_index()
    df.index = df.index.to_period('M').to_timestamp('M')
    return df

def resample_to_monthly(df, date_col, value_col, agg='mean'):
    df = to_monthly_index(df, date_col)
    monthly = df[[value_col]].resample('M').agg(agg)
    return monthly

print('Helpers defined.')

Helpers defined.


In [8]:
cache_file = CACHE / 'Saadaoui_2026_JCE.parquet'
REFRESH_CACHE = False

if cache_file.exists() and not REFRESH_CACHE:
    df = pd.read_parquet(cache_file)
    df = df.sort_values('Period').reset_index(drop=True)
    if 'Period_dt' not in df.columns:
        df['Period_dt'] = stata_month_to_datetime(df['Period'])
    print(f'Loaded from cache: {cache_file}')
else:
    if not DTA.exists():
        raise FileNotFoundError(f'Missing dataset: {DTA}')
    df = pd.read_stata(DTA)
    df = df.sort_values('Period').reset_index(drop=True)
    df['Period_dt'] = stata_month_to_datetime(df['Period'])
    cache_file.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(cache_file, index=False)
    print(f'Loaded from .dta and cached: {cache_file}')

# Derived columns
df['dllgop'] = D(df['llgop'])
df['dl2lgop'] = D(df['l2lgop'])
df['F2_d2pri'] = F(df['d2pri'], 2)

# Set monthly index for merging
df = df.set_index('Period_dt').sort_index()
df.index = df.index.to_period('M').to_timestamp('M')

BASE_CONTROLS = ['llwip', 'dllgop', 'l2lwip', 'dl2lgop']

print(f'Shape: {df.shape}')
print(f'Date range: {df.index.min().date()} to {df.index.max().date()}')
print(f'Base controls: {BASE_CONTROLS}')

# Verify baseline
f_base = first_stage_f(df, controls=BASE_CONTROLS)
print(f'\n=== STEP 0: BASELINE ===')
print(f'F-stat: {f_base:.3f} | {"PASS" if f_base > 150 else "FAIL"}')

Loaded from cache: C:\Users\HP\Desktop\replication+contribution\data\cache\Saadaoui_2026_JCE.parquet
Shape: (386, 52)
Date range: 1990-01-31 to 2022-02-28
Base controls: ['llwip', 'dllgop', 'l2lwip', 'dl2lgop']

=== STEP 0: BASELINE ===
F-stat: 236.185 | PASS


In [9]:
# Dictionary of all raw controls with their transforms
RAW_CONTROLS = {
    # Level controls (no log, no diff) — just resample to monthly
    'vix': {
        'file': 'vix.csv',
        'date_col': 'Date',
        'value_col': 'vix',
        'agg': 'mean',
        'log': False,
        'diff': False,
    },
    'gs10': {
        'file': 'gs10.csv',
        'date_col': 'observation_date',
        'value_col': 'GS10',
        'agg': 'last',
        'log': False,
        'diff': False,
    },
    'tb3ms': {
        'file': 'tb3ms.csv',
        'date_col': 'observation_date',
        'value_col': 'TB3MS',
        'agg': 'last',
        'log': False,
        'diff': False,
    },
    'tedrate': {
        'file': 'tedrate.csv',
        'date_col': 'observation_date',
        'value_col': 'TEDRATE',
        'agg': 'mean',
        'log': False,
        'diff': False,
    },
    'baa10y': {
        'file': 'baa10y.csv',
        'date_col': 'observation_date',
        'value_col': 'BAA10Y',
        'agg': 'last',
        'log': False,
        'diff': False,
    },
    'us_spread': {
        'file': 'us_spread.csv',
        'date_col': 'observation_date',
        'value_col': 'us_spread',
        'agg': 'last',
        'log': False,
        'diff': False,
    },
    'em_oas': {
        'file': 'em_oas.csv',
        'date_col': 'observation_date',
        'value_col': 'BAMLEMCBPIOAS',
        'agg': 'last',
        'log': False,
        'diff': False,
    },
    'us_bbb_oas': {
        'file': 'us_bbb_oas.csv',
        'date_col': 'observation_date',
        'value_col': 'BAMLC0A4CBBB',
        'agg': 'last',
        'log': False,
        'diff': False,
    },
    'us_hy_spread': {
        'file': 'us_credit_spread.csv',
        'date_col': 'observation_date',
        'value_col': 'BAMLH0A0HYM2',
        'agg': 'last',
        'log': False,
        'diff': False,
    },
    'us_ig_oas': {
        'file': 'us_ig_oas.csv',
        'date_col': 'observation_date',
        'value_col': 'BAMLC0A0CM',
        'agg': 'last',
        'log': False,
        'diff': False,
    },
    
    # Log + diff controls (prices, indices)
    'brent': {
        'file': 'brent_fred.csv',
        'date_col': 'observation_date',
        'value_col': 'DCOILBRENTEU',
        'agg': 'mean',
        'log': True,
        'diff': True,
    },
    'gold': {
        'file': 'gold_monthly.csv',
        'date_col': 'Date',
        'value_col': 'Price',
        'agg': 'last',  # already monthly
        'log': True,
        'diff': True,
    },
    'bdi': {
        'file': 'bdi_clean.csv',
        'date_col': 'date',
        'value_col': 'bdi',
        'agg': 'mean',
        'log': True,
        'diff': True,
    },
    'cny_usd': {
        'file': 'cny_usd.csv',
        'date_col': 'observation_date',
        'value_col': 'DEXCHUS',
        'agg': 'last',
        'log': True,
        'diff': True,
    },
    'dxy': {
        'file': 'dxy.csv',
        'date_col': 'observation_date',
        'value_col': 'DTWEXBGS',
        'agg': 'last',
        'log': True,
        'diff': True,
    },
    'em_fx': {
        'file': 'em_fx_idx.csv',
        'date_col': 'observation_date',
        'value_col': 'DTWEXEMEGS',
        'agg': 'last',
        'log': True,
        'diff': True,
    },
    'reer': {
        'file': 'reer_bis.csv',
        'date_col': 'observation_date',
        'value_col': 'RBUSBIS',
        'agg': 'last',  # already monthly
        'log': True,
        'diff': True,
    },
    'indpro': {
        'file': 'indpro.csv',
        'date_col': 'observation_date',
        'value_col': 'INDPRO',
        'agg': 'last',  # already monthly
        'log': True,
        'diff': True,
    },
    
    # Already monthly, no transform needed
    'gscpi': {
        'file': 'gscpi_cleaned.csv',
        'date_col': 'Date',
        'value_col': 'gscpi',
        'agg': 'last',
        'log': False,
        'diff': False,
    },
}

print(f'Defined {len(RAW_CONTROLS)} raw controls:')
for name, cfg in RAW_CONTROLS.items():
    transform = []
    if cfg['log']: transform.append('log')
    if cfg['diff']: transform.append('diff')
    tstr = '+'.join(transform) if transform else 'level'
    print(f'  {name:15s} ← {cfg["file"]:25s} | {tstr}')

Defined 19 raw controls:
  vix             ← vix.csv                   | level
  gs10            ← gs10.csv                  | level
  tb3ms           ← tb3ms.csv                 | level
  tedrate         ← tedrate.csv               | level
  baa10y          ← baa10y.csv                | level
  us_spread       ← us_spread.csv             | level
  em_oas          ← em_oas.csv                | level
  us_bbb_oas      ← us_bbb_oas.csv            | level
  us_hy_spread    ← us_credit_spread.csv      | level
  us_ig_oas       ← us_ig_oas.csv             | level
  brent           ← brent_fred.csv            | log+diff
  gold            ← gold_monthly.csv          | log+diff
  bdi             ← bdi_clean.csv             | log+diff
  cny_usd         ← cny_usd.csv               | log+diff
  dxy             ← dxy.csv                   | log+diff
  em_fx           ← em_fx_idx.csv             | log+diff
  reer            ← reer_bis.csv              | log+diff
  indpro          ← indpro.csv      

In [10]:
def load_and_transform(name, cfg):
    """Load a raw CSV, resample to monthly, apply log/diff transforms."""
    path = RAW / cfg['file']
    df_raw = pd.read_csv(path)
    
    # Convert to monthly
    df_monthly = resample_to_monthly(df_raw, cfg['date_col'], cfg['value_col'], cfg['agg'])
    
    # Rename column
    df_monthly = df_monthly.rename(columns={cfg['value_col']: name})
    
    # Apply transforms
    if cfg['log']:
        df_monthly[f'l{name}'] = np.log(df_monthly[name])
        if cfg['diff']:
            df_monthly[f'dl{name}'] = df_monthly[f'l{name}'].diff()
            return df_monthly[[f'dl{name}']].rename(columns={f'dl{name}': name})
        else:
            return df_monthly[[f'l{name}']].rename(columns={f'l{name}': name})
    else:
        if cfg['diff']:
            df_monthly[f'd{name}'] = df_monthly[name].diff()
            return df_monthly[[f'd{name}']].rename(columns={f'd{name}': name})
        else:
            return df_monthly[[name]]

def check_f(df_test, label, controls):
    f = first_stage_f(df_test, controls=controls)
    status = 'PASS' if f > 150 else 'FAIL'
    print(f'{label:40s} | F={f:8.2f} | {status}')
    return f

# Start with base dataframe
df_work = df.copy()
current_controls = BASE_CONTROLS.copy()
f_history = [('baseline', f_base)]

print(f'\n=== INCREMENTAL MACRO CONTROL ADDITION ===')
print(f'Base controls: {current_controls}')
print(f'Base F-stat:  {f_base:.3f}\n')

# Add each control one by one
for name, cfg in RAW_CONTROLS.items():
    try:
        df_ctrl = load_and_transform(name, cfg)
        df_merged = df_work.join(df_ctrl, how='left')
        
        # Add new control to list
        test_controls = current_controls + [name]
        
        # Check F-stat
        f_new = check_f(df_merged, f'STEP +{name}', test_controls)
        f_history.append((name, f_new))
        
        if f_new > 150:
            # Keep it
            df_work = df_merged.copy()
            current_controls = test_controls.copy()
            print(f'  → KEPT {name}')
        else:
            print(f'  → DROPPED {name} (F < 150)')
            
    except Exception as e:
        print(f'{name:40s} | ERROR: {e}')

print(f'\n=== FINAL MACRO SET ===')
print(f'Controls: {current_controls}')
print(f'Count: {len(current_controls)}')
f_final_macro = first_stage_f(df_work, controls=current_controls)
print(f'F-stat: {f_final_macro:.3f}')


=== INCREMENTAL MACRO CONTROL ADDITION ===
Base controls: ['llwip', 'dllgop', 'l2lwip', 'dl2lgop']
Base F-stat:  236.185

STEP +vix                                | F=  232.07 | PASS
  → KEPT vix
STEP +gs10                               | F=  231.44 | PASS
  → KEPT gs10
STEP +tb3ms                              | F=  231.19 | PASS
  → KEPT tb3ms
STEP +tedrate                            | F=  226.79 | PASS
  → KEPT tedrate
STEP +baa10y                             | F=  223.40 | PASS
  → KEPT baa10y
STEP +us_spread                          | F=  223.40 | PASS
  → KEPT us_spread
em_oas                                   | ERROR: zero-size array to reduction operation maximum which has no identity
us_bbb_oas                               | ERROR: zero-size array to reduction operation maximum which has no identity
us_hy_spread                             | ERROR: zero-size array to reduction operation maximum which has no identity
us_ig_oas                                | ERROR: zero-size 

In [11]:
NLP_CONTROLS = {
    'gpr': {'file': 'gpr_monthly.csv', 'date_col': 'date', 'value_col': 'gpr', 'shift': 1},
    'ea_gpr': {'file': 'ea_gpr_monthly.csv', 'date_col': 'date', 'value_col': 'ea_gpr', 'shift': 1},
    'wui': {'file': 'wui.csv', 'date_col': 'date', 'value_col': 'wui', 'shift': 1},
    'gdelt_events': {'file': 'gdelt_monthly.csv', 'date_col': 'date', 'value_col': 'events', 'shift': 1},
    'gdelt_sentiment': {'file': 'sentiment_monthly.csv', 'date_col': 'date', 'value_col': 'sentiment', 'shift': 1},
    'finbert': {'file': 'finbert_sentiment_monthly.csv', 'date_col': 'date', 'value_col': 'finbert', 'shift': 1},
}

print(f'\n=== INCREMENTAL NLP CONTROL ADDITION ===')
print(f'Current controls: {current_controls}')
print(f'Current F-stat:  {f_final_macro:.3f}\n')

for name, cfg in NLP_CONTROLS.items():
    path = NLP / cfg['file']
    if not path.exists():
        print(f'{name:40s} | FILE NOT FOUND: {path}')
        continue
    
    try:
        df_nlp = pd.read_csv(path, parse_dates=[cfg['date_col']])
        df_nlp = to_monthly_index(df_nlp, cfg['date_col'])
        df_nlp = df_nlp[[cfg['value_col']]].rename(columns={cfg['value_col']: name})
        df_nlp = df_nlp.shift(cfg['shift'])  # Look-ahead prevention
        
        df_merged = df_work.join(df_nlp, how='left')
        test_controls = current_controls + [name]
        
        f_new = check_f(df_merged, f'STEP +{name}', test_controls)
        f_history.append((name, f_new))
        
        if f_new > 150:
            df_work = df_merged.copy()
            current_controls = test_controls.copy()
            print(f'  → KEPT {name}')
        else:
            print(f'  → DROPPED {name} (F < 150)')
            
    except Exception as e:
        print(f'{name:40s} | ERROR: {e}')

print(f'\n=== FINAL FULL SET ===')
print(f'Controls ({len(current_controls)}): {current_controls}')
f_final = first_stage_f(df_work, controls=current_controls)
print(f'F-stat: {f_final:.3f}')


=== INCREMENTAL NLP CONTROL ADDITION ===
Current controls: ['llwip', 'dllgop', 'l2lwip', 'dl2lgop', 'vix', 'gs10', 'tb3ms', 'tedrate', 'baa10y', 'us_spread', 'brent', 'gold', 'bdi', 'cny_usd', 'em_fx', 'reer', 'gscpi']
Current F-stat:  155.437

gpr                                      | ERROR: Missing column provided to 'parse_dates': 'date'
ea_gpr                                   | ERROR: Missing column provided to 'parse_dates': 'date'
wui                                      | ERROR: Missing column provided to 'parse_dates': 'date'
gdelt_events                             | ERROR: Missing column provided to 'parse_dates': 'date'
gdelt_sentiment                          | ERROR: Missing column provided to 'parse_dates': 'date'
finbert                                  | ERROR: Missing column provided to 'parse_dates': 'date'

=== FINAL FULL SET ===
Controls (17): ['llwip', 'dllgop', 'l2lwip', 'dl2lgop', 'vix', 'gs10', 'tb3ms', 'tedrate', 'baa10y', 'us_spread', 'brent', 'gold', 'bdi'